# Day 1 Morning - 最簡單的 AI Agent

**課程：LLM Application - AI Agent**

---

## 1. 什麼是最簡單的 Agent？

一個 Agent 的核心就是三個東西：

```
Agent = LLM + Tools + Loop
```

最簡單的 Agent：給 LLM 幾個工具，讓它自己決定要用哪個，然後不斷循環直到任務完成。

```
使用者提出需求
    ↓
LLM 思考 → 決定呼叫哪個工具
    ↓
執行工具 → 回傳結果給 LLM
    ↓
LLM 繼續思考 → 決定下一步（呼叫工具 or 完成）
    ↓
重複直到 LLM 認為任務完成
```

**參考專案：** https://github.com/ywchiu/local_agentic_llm  
這個專案用最少的程式碼實作了一個 Agent，只給 LLM 4 個工具（write_file, read_file, run_command, list_files），就能讓它自主建構程式。

## 環境設定

In [ ]:
%%capture
!pip install openai

In [ ]:
import os
import json
import subprocess
import tempfile
from openai import OpenAI

# 設定 API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# 建立 OpenAI client
client = OpenAI()

# 建立工作目錄（用 temp 目錄避免污染環境）
WORKSPACE = tempfile.mkdtemp(prefix="agent_workspace_")
print(f"工作目錄: {WORKSPACE}")

## 2. 定義工具 (Tools)

我們給 Agent 4 個最基本的工具，讓它可以操作檔案系統和執行指令：

In [ ]:
# === 工具 1: 寫入檔案 ===
def write_file(path, content):
    """寫入檔案"""
    # 安全性：限制在工作目錄內
    full_path = os.path.join(WORKSPACE, path)
    os.makedirs(os.path.dirname(full_path) if os.path.dirname(full_path) else WORKSPACE, exist_ok=True)
    with open(full_path, 'w') as f:
        f.write(content)
    return f"File written: {full_path} ({len(content)} bytes)"

# === 工具 2: 讀取檔案 ===
def read_file(path):
    """讀取檔案"""
    full_path = os.path.join(WORKSPACE, path)
    with open(full_path, 'r') as f:
        return f.read()

# === 工具 3: 執行 shell 指令 ===
def run_command(command):
    """執行 shell 指令"""
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True,
        timeout=30, cwd=WORKSPACE
    )
    return result.stdout + result.stderr

# === 工具 4: 列出目錄下的檔案 ===
def list_files(path='.'):
    """列出目錄下的檔案"""
    full_path = os.path.join(WORKSPACE, path)
    return '\n'.join(os.listdir(full_path))

# 測試一下
print(write_file("test.txt", "Hello Agent!"))
print(read_file("test.txt"))
print(list_files())

## 3. 將工具轉換成 OpenAI Tool 格式

OpenAI 的 function calling 需要用 JSON Schema 描述每個工具的參數格式，LLM 才知道怎麼呼叫：

In [ ]:
# OpenAI function calling 的工具定義格式
tools = [
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "寫入內容到指定檔案路徑",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "檔案路徑"},
                    "content": {"type": "string", "description": "檔案內容"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "讀取指定檔案的內容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "檔案路徑"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "執行 shell 指令並回傳結果",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "要執行的 shell 指令"}
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "列出指定目錄下的所有檔案",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "目錄路徑，預設為當前目錄", "default": "."}
                },
                "required": []
            }
        }
    }
]

print(f"共定義了 {len(tools)} 個工具:")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

## 4. 建立 Agent 迴圈

Agent 的核心就是一個迴圈：
1. 把訊息送給 LLM
2. 如果 LLM 回傳 tool_calls → 執行工具 → 把結果加回訊息 → 回到 1
3. 如果 LLM 沒有 tool_calls → 任務完成，跳出迴圈

In [ ]:
# 工具名稱對應到實際的 Python 函數
tool_map = {
    "write_file": write_file,
    "read_file": read_file,
    "run_command": run_command,
    "list_files": list_files,
}

def execute_tool(name, args):
    """執行工具並回傳結果"""
    func = tool_map.get(name)
    if func is None:
        return f"Error: 未知的工具 '{name}'"
    try:
        return func(**args)
    except Exception as e:
        return f"Error: {str(e)}"


def run_agent(user_prompt, max_turns=10):
    """執行 Agent 迴圈"""
    print(f"{'='*60}")
    print(f"使用者需求: {user_prompt}")
    print(f"{'='*60}")

    messages = [
        {
            "role": "system",
            "content": "你是一個軟體工程師。用戶會請你建構東西。使用提供的工具來寫檔案和執行指令。完成後請總結你做了什麼。"
        },
        {"role": "user", "content": user_prompt}
    ]

    for turn in range(max_turns):
        print(f"\n--- 第 {turn + 1} 輪 ---")

        # 呼叫 LLM
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=messages,
            tools=tools,
            temperature=0
        )

        message = response.choices[0].message
        messages.append(message)

        # 如果沒有 tool_calls，表示 Agent 完成了
        if not message.tool_calls:
            print(f"\nAgent 完成！共 {turn + 1} 輪")
            print(f"最終回覆: {message.content}")
            break

        # 執行每個 tool call
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            # 印出工具呼叫（截斷太長的內容）
            args_display = {k: (v[:80] + '...' if isinstance(v, str) and len(v) > 80 else v) for k, v in args.items()}
            print(f"  [Tool] {name}({args_display})")

            result = execute_tool(name, args)
            print(f"  [Result] {str(result)[:200]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    return messages

## 5. 測試：讓 Agent 自己寫程式

我們給 Agent 一個任務，看它怎麼自主完成：

In [ ]:
# 讓 Agent 寫一個 CSV 轉 JSON 的程式
messages = run_agent("寫一個 Python 程式，可以將 CSV 檔案轉換成 JSON 格式，要能處理不同的分隔符號和缺失值")

In [ ]:
# 看看 Agent 建立了哪些檔案
print("工作目錄中的檔案:")
for f in os.listdir(WORKSPACE):
    full = os.path.join(WORKSPACE, f)
    size = os.path.getsize(full)
    print(f"  {f} ({size} bytes)")

## 6. 測試：讓 Agent 建立一個簡單的網頁

In [ ]:
# 讓 Agent 建立 HP 筆電產品頁面
messages = run_agent("建立一個簡單的 HTML 頁面，顯示 HP 筆電產品列表，包含 EliteBook、Pavilion、Spectre 三個系列的基本規格")

In [ ]:
# 顯示 Agent 產生的 HTML
from IPython.display import HTML

# 找出 HTML 檔案
html_files = [f for f in os.listdir(WORKSPACE) if f.endswith('.html')]
if html_files:
    html_content = open(os.path.join(WORKSPACE, html_files[0])).read()
    print(f"顯示: {html_files[0]}")
    display(HTML(html_content))
else:
    print("沒有找到 HTML 檔案")

## 7. Agent 的核心概念解析

回顧我們剛剛建立的 Agent，它的運作機制其實非常簡單：

### Tool Use（工具使用）
- LLM 透過 **function calling** 決定要使用哪個工具
- 我們定義好工具的名稱、描述、參數格式（JSON Schema）
- LLM 會根據任務需求，自動產生正確的函數呼叫

### Agent Loop（代理迴圈）
- 迴圈持續執行，直到 LLM 認為任務完成（不再呼叫工具）
- 每次迴圈：LLM 思考 → 呼叫工具 → 取得結果 → 繼續思考
- 設定 `max_turns` 防止無限迴圈

### Autonomy（自主性）
- LLM **自主決定**下一步行動：要用什麼工具、傳什麼參數
- 我們不需要寫 if-else 判斷邏輯，LLM 自己規劃執行步驟
- 這就是 Agent 和傳統程式最大的差別

### 這個簡單 Agent 的架構

```
┌─────────────────────────────────────┐
│           Agent Loop                │
│                                     │
│  User Prompt                        │
│       ↓                             │
│  ┌─────────┐    ┌──────────────┐    │
│  │   LLM   │───→│  Tool Calls  │    │
│  │(gpt-4.1)│    │              │    │
│  └────↑────┘    └──────┬───────┘    │
│       │                ↓            │
│       │         ┌──────────────┐    │
│       └─────────│ Tool Results │    │
│                 └──────────────┘    │
│                                     │
│  Tools: write_file, read_file,      │
│         run_command, list_files      │
└─────────────────────────────────────┘
```

## 8. 從簡單 Agent 到 LangGraph

我們剛剛建立的 Agent 雖然能動，但有幾個明顯的限制：

| 限制 | 說明 | LangGraph 解法 |
|------|------|----------------|
| **沒有狀態管理** | 只有 messages 列表，無法追蹤複雜狀態 | `StateGraph` 管理結構化狀態 |
| **沒有分支邏輯** | 只能線性執行，無法根據條件走不同路徑 | `add_conditional_edges` 條件路由 |
| **沒有記憶** | 重新執行就什麼都忘了 | `Checkpointer` 持久化記憶 |
| **沒有人類介入** | Agent 自己跑到底，無法中途確認 | `interrupt` 人機協作 |
| **沒有錯誤處理** | 工具失敗就 GG | 節點級別的錯誤處理和重試 |
| **沒有可觀測性** | 看不到 Agent 內部在幹嘛 | LangSmith 追蹤和除錯 |

### 下一步預告

接下來我們會用 **LangGraph** 來建構更強大的 Agent：

```python
# LangGraph 版本的 Agent（預告）
from langgraph.graph import StateGraph

graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("tools", execute_tools)
graph.add_conditional_edges("llm", should_continue)
graph.add_edge("tools", "llm")

agent = graph.compile(checkpointer=memory)
```

**核心觀念：** 今天的簡單 Agent 是所有 AI Agent 的基礎。理解了 LLM + Tools + Loop，接下來學 LangGraph 就是在這個基礎上加入更多工程化的功能。

In [ ]:
# 清理工作目錄
import shutil
shutil.rmtree(WORKSPACE, ignore_errors=True)
print("工作目錄已清理")